# Detector v2 (CenterNet r34, thạch bản + STT) — huấn luyện trên Kaggle · Run All · reset-safe

**Trước khi chạy:** Settings → Accelerator **GPU T4** (x2 cũng được; P100 có thể lỗi với torch mới) · Persistence **Files only**
(giữ `/kaggle/working/last.pt` khi restart) · Add Input → dataset dựng từ `i5v2_bundle.zip` (`make_bundle.py`).
Internet chỉ cần nếu đẩy lên HF (`HF_REPO` + Secret `HF_TOKEN`).

Mỗi epoch: train cân bằng 50/50 thạch bản/STT (aug nền xám · otsu/stretch · kéo dọc ±10 % · nét · blur · crop cột) →
đánh giá trên **val page-disjoint**: ok50 / miss / extra / |dy| / **% tầng n==N** (hộp thô @0,15 & 0,2) / cắt thân chữ,
**STT F1** (guard ≥ v1 − 0,01), Chrestomathie (sách không train). Best theo ok50_litho qua guard; dừng sớm 4 epoch.
Kết quả: `/kaggle/working/{best.pt, last.pt, metrics.csv, report.md}`. **Reset?** Run All lại → tự resume từ `last.pt`.

In [ ]:
# ---- 1) Cấu hình — chỉ sửa ở đây -------------------------------------------------
EPOCHS   = 20        # ≈ 2–3 phút/epoch trên T4 ở img 1024 (20 epoch ≈ 1 giờ + eval)
BATCH    = 4         # hết RAM GPU -> 2
IMG      = 1024      # hoặc 1280 (chậm ~1,6×, batch 2)
LR       = 2e-4      # cosine + warmup 1 epoch
SEED     = 0
EVAL_CHRESTO = 20    # số trang Chrestomathie (held-out, không train) đo mỗi epoch; 0 = bỏ
HF_REPO  = ""        # vd "mdnt571/nom-char-det-v2": đẩy last/best mỗi epoch + resume từ hub (cần Secret HF_TOKEN). "" = local
EXTRA    = ""        # tham số thêm cho train_kaggle.py, vd "--patience 6 --stt-tol 0.01 --p-gray 0.5"
SMOKE    = False     # True: chạy thử 4 trang/miền, 1 epoch (kiểm tra đường ống ~2 phút)

In [ ]:
# ---- 2) Tìm bundle, chép mã ra /kaggle/working, kiểm GPU ---------------------------
import os, sys, glob, shutil, subprocess
hits = sorted(glob.glob("/kaggle/input/**/train_kaggle.py", recursive=True)) or sorted(glob.glob("./**/train_kaggle.py", recursive=True))
assert hits, "Không thấy train_kaggle.py — Add Input: gắn dataset dựng từ i5v2_bundle.zip"
DATA = os.path.dirname(hits[0])
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work")
CODE = os.path.join(WORK, "i5v2_code"); os.makedirs(CODE, exist_ok=True)
shutil.copytree(os.path.join(DATA, "i5v2"), os.path.join(CODE, "i5v2"), dirs_exist_ok=True)
shutil.copy2(os.path.join(DATA, "train_kaggle.py"), os.path.join(CODE, "train_kaggle.py"))
for f in ("manifest_train.json", "manifest_val.json", "v1/detector_r34.best.pt", "bundle_stats.json"):
    assert os.path.exists(os.path.join(DATA, f)), f"bundle thiếu {f}"
tok = ""
if HF_REPO:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "huggingface_hub"], check=False)
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        tok = os.environ.get("HF_TOKEN", "")
    os.environ["HF_TOKEN"] = tok or ""
import torch, json
print("data :", DATA); print("code :", CODE); print("out  :", WORK)
print("GPU  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (bật GPU T4!)", "| torch", torch.__version__)
print("bundle:", json.load(open(os.path.join(DATA, "bundle_stats.json")))["by_domain_split"])
print("HF   :", HF_REPO or "(local-only)", "| token", "có" if tok else "không")
print("resume:", "có last.pt → tiếp tục" if os.path.exists(os.path.join(WORK, "last.pt")) else "chưa có last.pt → từ đầu")

In [ ]:
# ---- 3) Train — reset-safe (chạy lại cell này/Run All sau reset để tiếp tục) --------
cmd = [sys.executable, os.path.join(CODE, "train_kaggle.py"), "--data", DATA, "--out", WORK,
       "--epochs", str(EPOCHS), "--batch", str(BATCH), "--img", str(IMG), "--lr", str(LR), "--seed", str(SEED),
       "--eval-chresto", str(EVAL_CHRESTO), "--workers", "2"]
if HF_REPO: cmd += ["--hf-repo", HF_REPO]
if SMOKE:   cmd += ["--smoke"]
if EXTRA:   cmd += EXTRA.split()
print("$", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
# ---- 4) Kết quả: bảng v1 vs v2 + theo epoch ------------------------------------------
import csv
print(open(os.path.join(WORK, "report.md"), encoding="utf-8").read())
rows = list(csv.DictReader(open(os.path.join(WORK, "metrics.csv"), encoding="utf-8")))
keys = ["epoch", "loss", "litho_ok50", "litho_tiers_eq_015", "litho_tiers_eq_020", "litho_cut", "stt_F1_020", "chresto_tiers_eq_015", "is_best"]
print(" | ".join(keys))
for r in rows:
    print(" | ".join(str(r.get(k, "")) for k in keys))
bp = os.path.join(WORK, "best.pt")
if os.path.exists(bp):
    d = torch.load(bp, map_location="cpu", weights_only=False)
    print("\nbest.pt: epoch", d["epoch"], "| img", d["img"], "| use_dcn", d["use_dcn"], "|", os.path.getsize(bp) // 2**20, "MB")
    print("val:", {k: d["val"].get(k) for k in ("litho_ok50", "litho_tiers_eq_015", "litho_cut", "stt_F1_020")})
    print("v1 :", {k: d["base_v1"].get(k) for k in ("litho_ok50", "litho_tiers_eq_015", "litho_cut", "stt_F1_020")})
else:
    print("\nKHÔNG có best.pt: không epoch nào qua guard STT F1 ≥ v1 − 0,01 → giữ v1 (xem report.md)")

### Xong
- Tải **`best.pt`** (tab Output / `/kaggle/working/best.pt`) + `metrics.csv` + `report.md` về máy.
- Ở repo: `lab/i5_detector_v2/apply_v2.sh <đường dẫn best.pt>` → đo `box_ref_eval` v1↔v2 trên ảnh gốc, build `--book all-new --suffix _v2`.
- Reset / hết giờ: **Run All** lại — cell (3) tự resume từ `last.pt` (Persistence Files only) hoặc từ HF nếu khai `HF_REPO`.